#  TP 4 : Étude empirique : les pratiques de refactoring avec les agents IA
## MGL804 – Réalisation et maintenance des logiciels

**Objectifs :**
1. **Partie 1** — Quantifier la prévalence des suggestions de refactoring dans les commentaires de révision humains sur des PRs agentiques.
2. **Partie 2** — Vérifier avec RefactoringMiner si les agents appliquent ces suggestions.

**Jeu de données :** [AIDev](https://huggingface.co/datasets/hao-li/AIDev) (sous-ensemble AIDev-pop)

> ⚠️ Ce notebook est un **guide de laboratoire**. Il fournit la structure, des indices et des points de départ — **c'est à vous d'écrire le code**. Les cellules marquées `# TODO` ou `# VOTRE CODE ICI` sont à compléter.\n",

> 🚨 **Avertissement important :** Les indices fournis dans ce notebook (noms de colonnes, noms de variables, exemples de jointures) sont indicatifs et peuvent ne pas correspondre exactement à la structure réelle des données. **Vous devez explorer les tables vous-même, vérifier les noms de colonnes réels avec `.columns` ou `.head()`, et adapter votre code en conséquence — ne copiez pas les indices sans les valider au préalable.**

---
## 0. Installation et configuration

Exécutez cette cellule une seule fois pour installer les dépendances.

In [ ]:
# Décommentez si nécessaire :
# !pip install pandas pyarrow matplotlib seaborn scipy tqdm datasets nbformat

### 0.1 Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import json
import subprocess
import os
from tqdm.auto import tqdm
from scipy import stats

pd.set_option('display.max_columns', 30)
pd.set_option('display.max_colwidth', 120)
sns.set_theme(style="whitegrid")

print("Imports réussis")

### 0.2 Configuration de Java et RefactoringMiner

RefactoringMiner 3.x requiert **Java 17+**. Si votre serveur a une version plus ancienne, installez-la via conda.

> **Indice :** Vérifiez votre version Java avec `!java -version`. Si c'est Java 8 ou 11, vous devez installer Java 17.

In [ ]:
# Vérifiez votre version Java actuelle
!java -version

In [ ]:
# Si Java < 17, décommentez et exécutez :
# !conda install -c conda-forge openjdk=17 -y

# Puis configurez le PATH :
# import os
# conda_base = os.popen('conda info --base').read().strip()
# java17_home = f"{conda_base}/lib/jvm"
# os.environ['JAVA_HOME'] = java17_home
# os.environ['PATH'] = f"{java17_home}/bin:{os.environ['PATH']}"
# !java -version

In [ ]:
# Token GitHub pour l'API (nécessaire pour le mode -gc / -gp de RefactoringMiner)
# Générez un token : GitHub → Settings → Developer settings → Personal access tokens
#
# INDICE : Deux façons de le fournir :
#   1) Variable d'environnement :  os.environ['GITHUB_OAUTH'] = 'ghp_...'
#   2) Fichier properties :        echo "oauth=ghp_..." > github-oauth.properties

# VOTRE CODE ICI
# os.environ['GITHUB_OAUTH'] = 'ghp_VOTRE_TOKEN'

---
## 1. Chargement des données AIDev

Nous chargeons les tables Parquet directement depuis Hugging Face.

> **Indice :** Après le premier chargement, sauvegardez localement avec `df.to_parquet("local_file.parquet")` pour accélérer les exécutions suivantes.

In [ ]:
print("Chargement des pull requests...")
df_prs = pd.read_parquet("hf://datasets/hao-li/AIDev/pull_request.parquet")

print("Chargement des commentaires de révision (v2)...")
df_review_comments = pd.read_parquet("hf://datasets/hao-li/AIDev/pr_review_comments_v2.parquet")

print("Chargement des commits par PR...")
df_commits = pd.read_parquet("hf://datasets/hao-li/AIDev/pr_commits.parquet")

print("Chargement des détails de commits...")
df_commit_details = pd.read_parquet("hf://datasets/hao-li/AIDev/pr_commit_details.parquet")

print("Chargement de la timeline...")
df_timeline = pd.read_parquet("hf://datasets/hao-li/AIDev/pr_timeline.parquet")

print("Chargement des reviews...")
df_reviews = pd.read_parquet("hf://datasets/hao-li/AIDev/pr_reviews.parquet")

print("Chargement des dépôts...")
df_repos = pd.read_parquet("hf://datasets/hao-li/AIDev/repository.parquet")

print("\n✅ Toutes les tables sont chargées.")

### 1.1 Exploration rapide des données

Prenez le temps d'explorer la structure de chaque table avant de commencer l'analyse.

> **Indice :** Pour chaque table, examinez les colonnes, le nombre de lignes, et quelques lignes avec `.head()`. Identifiez les clés de jointure potentielles (`id`, `pr_id`, `repo_id`, etc.).

In [ ]:
# Aperçu des tables et de leurs colonnes
tables = {
    "pull_request": df_prs,
    "pr_review_comments_v2": df_review_comments,
    "pr_commits": df_commits,
    "pr_commit_details": df_commit_details,
    "pr_timeline": df_timeline,
    "pr_reviews": df_reviews,
    "repository": df_repos,
}

for name, df in tables.items():
    print(f"\n{'='*60}")
    print(f"{name} — {len(df):,} lignes, {len(df.columns)} colonnes")
    print(f"   Colonnes : {list(df.columns)}")

In [ ]:
# TODO : Explorez les tables individuellement
# Essayez : df_prs.head(3), df_review_comments.head(3), df_repos.head(3)
# Repérez les colonnes clés pour les jointures à venir.

# VOTRE CODE ICI

---
## Partie 1 : Prévalence des suggestions de refactoring (13 pts)

### Étape 1.1 — Filtrer les PRs agentiques

Les 5 agents connus dans AIDev sont : **Codex, Devin, Copilot, Cursor, Claude Code**.

> **Indice :** Cherchez une colonne dans `df_prs` qui identifie l'agent (ex. `'agent'`). Affichez ses valeurs uniques avec `.unique()` pour connaître les noms exacts, puis filtrez avec `.isin()`.

In [ ]:
# TODO : Identifiez la colonne qui indique l'agent et affichez ses valeurs uniques

# VOTRE CODE ICI

In [ ]:
# TODO : Filtrez les PRs agentiques en utilisant les noms exacts trouvés ci-dessus
#
# INDICE :
#   AGENTS = ['OpenAI_Codex', 'Copilot', 'Devin', 'Cursor', 'Claude_Code']  # vérifiez les noms exacts
#   df_agentic = df_prs[df_prs['agent'].isin(AGENTS)]
#
# Affichez le nombre de PRs par agent avec .value_counts()

# VOTRE CODE ICI

### Étape 1.2 — Joindre avec les commentaires de révision

Fusionnez les PRs agentiques avec les commentaires inline. **Gardez tous les commentaires** (humains et bots), puis analysez la distribution des auteurs (humain vs IA/bot).

> **Indice :** Il n'y a pas de clé directe entre `df_review_comments` et `df_prs`. Vous devez passer par `df_reviews` comme table intermédiaire :
> 1. `df_review_comments` → `df_reviews` (via `pull_request_review_id` ↔ `id`)
> 2. Résultat → `df_prs` (via `pr_id` ↔ `id`)
>
> Utilisez `.merge()` avec le paramètre `suffixes` pour éviter les conflits de noms de colonnes.
>
> Après la jointure, explorez la colonne auteur pour distinguer les commentaires humains des commentaires bot/IA et affichez la distribution (ex. `.value_counts()`).

In [ ]:
# TODO : Trouvez les colonnes communes entre les tables
print("Colonnes PR      :", sorted(df_prs.columns))
print("Colonnes Reviews :", sorted(df_reviews.columns))
print("Colonnes Comments:", sorted(df_review_comments.columns))

In [ ]:
# TODO : Effectuez la jointure en deux étapes
#
# INDICE :
#   Étape A : review_comments + reviews
#     df_merged = df_review_comments.merge(df_reviews,
#         left_on='pull_request_review_id', right_on='id',
#         how='left', suffixes=('_review_comment', '_review'))
#
#   Étape B : résultat + pull_request
#     df_comments = df_merged.merge(df_prs,
#         left_on='pr_id', right_on='id',
#         how='left', suffixes=('_comment', '_pr'))
#
# Vérifiez avec df_comments.shape et df_comments.head(3)

# VOTRE CODE ICI

In [ ]:
# TODO : Analysez la distribution des commentaires humains vs bots/IA
#
# INDICES :
#   1. Cherchez la colonne auteur : [c for c in df_comments.columns if 'user' in c.lower() or 'author' in c.lower()]
#   2. Identifiez les bots (souvent le nom contient '[bot]' ou 'github-actions')
#   3. Affichez la distribution : combien de commentaires humains vs bots ?
#   4. Visualisez avec un barplot ou un camembert
#
# NOTE : Gardez TOUS les commentaires dans df_comments pour la suite de l'analyse.

# VOTRE CODE ICI

### Étape 1.3 — Identifier les commentaires de refactoring (mots-clés)

Appliquez le dictionnaire de mots-clés fourni dans l'énoncé.

> **Indice :** Combinez les regex en un seul pattern avec `'|'.join(patterns)`, puis appliquez `.str.contains()` sur la colonne body des commentaires. Attention au paramètre `na=False` pour gérer les valeurs manquantes.

In [ ]:
import re

# Patterns de détection de refactoring (fournis dans l'énoncé)
KEYWORD_PATTERNS = {
    "Add*"                         : r"\badd\w*",
    "Chang*"                       : r"\bchang\w*",
    "Chang* the name"              : r"\bchang\w*\s+the\s+name",
    "Cleanup"                      : r"\bcleanup\b",
    "Clean* up"                    : r"\bclean\w*\s+up",
    "Code clarity"                 : r"\bcode\s+clarity",
    "Code clean*"                  : r"\bcode\s+clean\w*",
    "Code organization"            : r"\bcode\s+organization",
    "Code review"                  : r"\bcode\s+review",
    "Clean code"                   : r"\bclean\s+code",
    "Creat*"                       : r"\bcreat\w*",
    "Customiz*"                    : r"\bcustomiz\w*",
    "Easier to maintain"           : r"\beasier\s+to\s+maintain",
    "Encapsulat*"                  : r"\bencapsulat\w*",
    "Enhanc*"                      : r"\benhanc\w*",
    "Extend*"                      : r"\bextend\w*",
    "Extract*"                     : r"\bextract\w*",
    "Fix*"                         : r"\bfix\w*",
    "Inlin*"                       : r"\binlin\w*",
    "Improv*"                      : r"\bimprov\w*",
    "Improv* code quality"         : r"\bimprov\w*\s+code\s+quality",
    "Introduc*"                    : r"\bintroduc\w*",
    "Merg*"                        : r"\bmerg\w*",
    "Modif*"                       : r"\bmodif\w*",
    "Modulariz*"                   : r"\bmodulariz\w*",
    "Migrat*"                      : r"\bmigrat\w*",
    "Mov*"                         : r"\bmov\w*",
    "Organiz*"                     : r"\borganiz\w*",
    "Polish*"                      : r"\bpolish\w*",
    "Reduc*"                       : r"\breduc\w*",
    "Refactor*"                    : r"\brefactor\w*",
    "Refin*"                       : r"\brefin\w*",
    "Remov*"                       : r"\bremov\w*",
    "Remov* redundant code"        : r"\bremov\w*\s+redundant\s+code",
    "Renam*"                       : r"\brenam\w*",
    "Remov* unused dependencies"   : r"\bremov\w*\s+unused\s+dependenc\w*",
    "Reorganiz*"                   : r"\breorganiz\w*",
    "Replac*"                      : r"\breplac\w*",
    "Restructur*"                  : r"\brestructur\w*",
    "Rework*"                      : r"\brework\w*",
    "Rewrit*"                      : r"\brewrit\w*",
    "Simplif*"                     : r"\bsimplif\w*",
    "Split*"                       : r"\bsplit\w*",
}

combined_pattern = '|'.join(KEYWORD_PATTERNS.values())
print(f"Pattern combiné ({len(KEYWORD_PATTERNS)} mots-clés)")

In [ ]:
# TODO : Appliquez le pattern sur le body des commentaires
#
# INDICES :
#   1. Identifiez la bonne colonne body : [c for c in df_comments.columns if 'body' in c.lower()]
#   2. Créez une colonne booléenne 'is_refactoring' avec .str.contains()
#   3. Affichez le nombre et pourcentage de commentaires de refactoring

# VOTRE CODE ICI

### Étape 1.4 — Calculer la prévalence par agent

> **Indice :** Utilisez `.groupby('agent').agg()` pour compter le total et la somme des `is_refactoring`, puis calculez le pourcentage.

In [ ]:
# TODO : Tableau récapitulatif de prévalence par agent
#
# INDICE :
#   prevalence = df_comments.groupby('agent').agg(
#       total_comments=('is_refactoring', 'count'),
#       refactoring_comments=('is_refactoring', 'sum'),
#   )
#   prevalence['percentage'] = ...

# VOTRE CODE ICI

### Étape 1.5 — Taxonomie et distribution par catégorie

Classifiez chaque commentaire de refactoring dans l'une des 6 catégories.

> **Indice :** Créez un dictionnaire `CATEGORIES` qui associe chaque catégorie à une liste de regex. Écrivez une fonction `classify_refactoring(text)` qui retourne la première catégorie correspondante, ou `'Autre'` si aucune ne correspond. Appliquez-la avec `.apply()`.
>
> Catégories suggérées : Changements structurels, Nommage/lisibilité, Code mort/nettoyage, Organisation du code, Duplication, Remplacement/migration.

In [ ]:
# TODO : Définissez les catégories et leurs patterns
#
# INDICE :
#   CATEGORIES = {
#       'Changements structurels': [r'\bextract\b', r'\binline\b', r'\bsplit\b', ...],
#       'Nommage / lisibilité':    [r'\brename', r'\bclarity', ...],
#       ...
#   }
#
#   def classify_refactoring(text):
#       # Itérez sur CATEGORIES, retournez la première catégorie matchée
#       ...
#
#   df_refactoring = df_comments[df_comments['is_refactoring']].copy()
#   df_refactoring['category'] = df_refactoring[body_col].apply(classify_refactoring)

# VOTRE CODE ICI

In [ ]:
# TODO : Diagramme en barres — distribution par catégorie
#
# INDICE : df_refactoring['category'].value_counts().plot(kind='barh')

# VOTRE CODE ICI

### Étape 1.6 — Analyse par agent et par langage

> **Indice :** Joignez `df_refactoring` avec `df_repos` pour récupérer la colonne `language`. Utilisez `pd.crosstab()` pour un tableau croisé agent × langage. Visualisez avec un barplot empilé.

In [ ]:
# TODO : Analyse par agent et par langage
#
# INDICES :
#   1. Joignez df_refactoring avec df_repos sur repo_id
#   2. pd.crosstab(df_refactoring_lang['agent'], df_refactoring_lang['language'])
#   3. Visualisez les 8 langages les plus fréquents avec .plot(kind='bar', stacked=True)

# VOTRE CODE ICI

### Étape 1.7 — Validation manuelle (précision / rappel)

Les commentaires à annoter vous seront envoyés sur Discord.

> **Indice :**
> 1. Exportez en CSV votre annotation manuelle
> 2. Après annotation, calculez : `precision = TP / (TP + FP)` et `recall = TP / (TP + FN)`

In [ ]:
# TODO : Échantillonnage et validation manuelle
#
# INDICES :
#   sample_positive = df_comments[df_comments['is_refactoring']].sample(50, random_state=42)
#   sample_negative = df_comments[~df_comments['is_refactoring']].sample(50, random_state=42)
#   sample_positive[[body_col]].to_csv('validation_positive.csv', index=False)
#
#   Après annotation :
#   TP = ...  # vrais positifs
#   FP = ...  # faux positifs
#   FN = ...  # faux négatifs
#   precision = TP / (TP + FP)
#   recall    = TP / (TP + FN)

# VOTRE CODE ICI

---
## Partie 2 : Application des suggestions avec RefactoringMiner (12 pts + 2 bonus)

> **Prérequis :** RefactoringMiner doit être installé et configuré (voir l'énoncé).
> Cette partie se concentre sur les projets **Java** et **Python** (supportés par RefactoringMiner 3.0).

### Étape 2.1 — Filtrer les projets Java et Python

> **Indice :** Filtrez `df_repos` par langage, puis joignez avec `df_agentic` pour obtenir les PRs agentiques dans ces langages.

In [ ]:
# TODO : Filtrer les dépôts par langage et joindre avec les PRs agentiques
#
# INDICES :
#   1. df_repos['language'].value_counts().head(10)  # voir les langages disponibles
#   2. df_repos_jp = df_repos[df_repos['language'].isin(['Java', 'Python'])]
#   3. df_prs_jp = df_agentic.merge(df_repos_jp[['id', 'language', 'full_name']], ...)
#   4. Affichez un tableau croisé language × agent

# VOTRE CODE ICI

### Étape 2.2 — Sélectionner les PRs avec commentaires de refactoring

> **Indice :** Récupérez les `pr_id` uniques de `df_refactoring`, puis filtrez `df_prs_jp` pour ne garder que les PRs qui ont au moins un commentaire de refactoring.

In [ ]:
# TODO : Ne garder que les PRs avec au moins 1 commentaire de refactoring
#
# INDICE :
#   pr_ids_with_refactoring = set(df_refactoring['pr_id'].unique())
#   df_prs_refactoring = df_prs_jp[df_prs_jp['id_pr'].isin(pr_ids_with_refactoring)]
#   # Attention au nom de la colonne id après la jointure (peut être 'id_pr')

# VOTRE CODE ICI

### Étape 2.3 — Construire la chronologie par PR

Pour chaque PR, identifiez les **commits de suivi** : commits effectués **après** un commentaire de refactoring.

```
Temps ──────────────────────────────────────────────>

  commit_1     comment_refactoring     commit_2 (suivi)     commit_3 (suivi)
     │                 │                    │                     │
     ▼                 ▼                    ▼                     ▼
  ─────────────────────────────────────────────────────────────────
```

> **Indices :**
> - La table `pr_timeline` contient les événements : `event='committed'` pour les commits, avec `commit_id` = SHA et `created_at` = timestamp
> - Certains événements `committed` n'ont pas de `created_at` → utilisez `.bfill()` pour propager le timestamp du prochain événement
> - Comparez les timestamps des commentaires de refactoring avec ceux des commits pour identifier les commits de suivi

In [ ]:
# TODO : Préparer la timeline (événements 'committed' uniquement)
#
# INDICE :
#   df_tl_commits = df_timeline[df_timeline['event'] == 'committed'].copy()
#   df_tl_commits['created_at'] = pd.to_datetime(df_tl_commits['created_at'], utc=True, errors='coerce')
#   Explorez avec : df_tl_commits[['pr_id', 'event', 'commit_id', 'created_at']].head(5)

# VOTRE CODE ICI

In [ ]:
# TODO : Écrire une fonction get_followup_commits(pr_id, df_ref_comments, df_pr_timeline)
#
# La fonction doit :
#   1. Récupérer la timeline de la PR
#   2. Propager les timestamps manquants avec bfill()
#   3. Pour chaque commentaire de refactoring de cette PR,
#      trouver les commits dont le timestamp est postérieur
#   4. Retourner une liste de dicts avec : pr_id, comment_id, comment_path,
#      followup_shas, n_followups
#
# INDICE pour la comparaison temporelle :
#   followups = tl_commits[tl_commits['effective_ts'] > comment_timestamp]
#   followup_shas = list(followups['commit_id'].dropna())

# VOTRE CODE ICI

### Étape 2.4 — Exécuter RefactoringMiner

> **Indices :**
> - Le chemin vers l'exécutable dépend de votre installation. Vérifiez avec `!ls /chemin/vers/RefactoringMiner`
> - Mode `-gc` : analyse un commit distant (nécessite token GitHub)
> - Mode `-gp` : analyse une PR entière (tous les commits)
> - Commencez par tester avec un petit échantillon (`SAMPLE_SIZE = 5`)
> - Sauvegardez les résultats intermédiaires en JSON pour éviter de tout relancer

In [ ]:
# TODO : Configurez le chemin vers RefactoringMiner et testez-le
#
# INDICE :
#   RMINER_PATH = '/chemin/vers/RefactoringMiner/bin/RefactoringMiner'
#   OUTPUT_DIR = 'rminer_outputs'
#   os.makedirs(OUTPUT_DIR, exist_ok=True)
#
# Test rapide :
#   cmd = [RMINER_PATH, '-gc',
#          'https://github.com/danilofes/refactoring-toy-example.git',
#          '36287f7c3b09eff78395267a3ac0d7da067863fd',
#          '10', '-json', 'test_output.json']
#   result = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
#   print("Return code:", result.returncode)

# VOTRE CODE ICI

In [ ]:
# TODO : Écrire les fonctions run_refactoringminer_commit() et run_refactoringminer_pr()
#
# INDICES pour run_refactoringminer_commit(repo_full_name, sha, timeout) :
#   1. Construisez l'URL : f"https://github.com/{repo_full_name}.git"
#   2. Définissez le fichier de sortie : os.path.join(OUTPUT_DIR, f"commit_{sha[:12]}.json")
#   3. Commande : [RMINER_PATH, '-gc', repo_url, sha, str(timeout), '-json', out_file]
#   4. Exécutez avec subprocess.run() et timeout
#   5. Si le fichier existe, chargez-le avec json.load()
#   6. Gérez les exceptions TimeoutExpired et autres
#
# Pour run_refactoringminer_pr(), c'est similaire avec '-gp' et le numéro de PR

# VOTRE CODE ICI

In [ ]:
# TODO : Boucle d'analyse sur les PRs sélectionnées
#
# INDICES :
#   - Commencez avec SAMPLE_SIZE = 5 pour tester
#   - Utilisez tqdm pour suivre la progression
#   - Sauvegardez en JSON toutes les 10 PRs (checkpoint)
#   - Stockez les résultats dans un dict : rminer_results[pr_number] = result
#
# ATTENTION : Respectez les limites de l'API GitHub (~5000 requêtes/heure)

# VOTRE CODE ICI

### Étape 2.5 — Croiser les résultats et classifier

Pour chaque commentaire de refactoring, vérifiez si un refactoring a été détecté dans un commit de suivi touchant le même fichier.

> **Indice — Schéma de classification :**
> ```
> Pour chaque commentaire de refactoring :
>   1. Y a-t-il des commits de suivi ?
>      NON → "PR abandonnée" (si non fusionnée) ou "Non appliqué"
>   2. RefactoringMiner détecte-t-il un refactoring dans le même fichier ?
>      OUI → "Appliqué (confirmé)"
>      NON → "Appliqué (non confirmé)" ou "Non appliqué"
> ```
>
> Dans la sortie RefactoringMiner, les fichiers sont dans `leftSideLocations[0].filePath` et `rightSideLocations[0].filePath`.

In [ ]:
# TODO : Écrire classify_application(comment_path, followup_commits, rminer_results, pr_merged)
#
# INDICES :
#   1. Si pas de followup_commits → 'PR abandonnée' ou 'Non appliqué'
#   2. Pour chaque SHA de suivi, cherchez les refactorings dans rminer_results
#   3. Comparez comment_path avec les filePath dans les refactorings détectés
#   4. Retournez la catégorie appropriée

# VOTRE CODE ICI

In [ ]:
# TODO : Appliquer la classification sur tous les commentaires de refactoring
#
# INDICES :
#   1. Construisez un index {sha: [refactorings]} depuis rminer_results
#   2. Pour chaque PR analysée, appelez get_followup_commits()
#   3. Pour chaque commentaire, appelez classify_application()
#   4. Stockez dans une liste de dicts → DataFrame df_results
#   5. Affichez df_results['classification'].value_counts()

# VOTRE CODE ICI

### Étape 2.6 — Calculer les taux d'application

> **Indice :** Utilisez `pd.crosstab()` pour un tableau agent × classification. Normalisez par ligne pour obtenir des pourcentages. Visualisez avec un barplot empilé.

In [ ]:
# TODO : Diagramme en barres empilées par agent + comparaison Java vs Python
#
# INDICES :
#   ct = pd.crosstab(df_results['agent'], df_results['classification'])
#   ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
#   ct_pct.plot(kind='bar', stacked=True, colormap='Set2')

# VOTRE CODE ICI

### Étape 2.7 (Bonus) — Correspondance sémantique

Pour les cas « appliqué confirmé », vérifiez si le type de refactoring détecté correspond à la suggestion.

> **Indice :** Créez un mapping entre les mots-clés de la suggestion et les types RefactoringMiner. Par exemple, le mot `'extract'` dans un commentaire devrait correspondre à `'Extract Method'`, `'Extract Class'`, etc. dans les résultats.

In [ ]:
# TODO (Bonus) : Vérifier la correspondance sémantique
#
# INDICE :
#   SUGGESTION_TO_RMINER = {
#       'extract': ['Extract Method', 'Extract Class', ...],
#       'rename':  ['Rename Method', 'Rename Variable', ...],
#       'inline':  ['Inline Method', 'Inline Variable'],
#       ...
#   }
#
#   def check_semantic_match(comment_body, detected_type):
#       # Vérifiez si un mot-clé du commentaire correspond au type détecté
#       ...

# VOTRE CODE ICI

---
## Annexe : Fonctions utilitaires

Quelques fonctions utiles que vous pouvez réutiliser.

In [ ]:
def safe_json_load(filepath):
    """Charge un fichier JSON en gérant les erreurs."""
    try:
        with open(filepath) as f:
            return json.load(f)
    except (json.JSONDecodeError, FileNotFoundError) as e:
        print(f"Erreur chargement {filepath}: {e}")
        return None


def extract_refactorings_from_rminer(result_json):
    """
    Extrait la liste plate de refactorings depuis la sortie RefactoringMiner.

    Returns:
        list of dict: [{sha, type, description, filePath}, ...]
    """
    refactorings = []
    if not result_json:
        return refactorings
    for commit in result_json.get('commits', []):
        sha = commit.get('sha1', '')
        for ref in commit.get('refactorings', []):
            left_files = [loc.get('filePath', '') for loc in ref.get('leftSideLocations', [])]
            right_files = [loc.get('filePath', '') for loc in ref.get('rightSideLocations', [])]
            refactorings.append({
                'sha': sha,
                'type': ref.get('type', ''),
                'description': ref.get('description', ''),
                'leftFiles': left_files,
                'rightFiles': right_files,
            })
    return refactorings

print("Fonctions utilitaires chargées.")